# 26. TF-IDF

## Overview
Term Frequency–Inverse Document Frequency (TF-IDF) weights terms according to how often they appear in a document and how widely they are distributed across the corpus. Terms common to every document generally receive less weight than terms that distinguish a document.

This notebook explores TF-IDF values, normalization, n-grams, and practical usage with scikit-learn.

## 1. Import libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Create a document collection

In [2]:
documents = [
    "data science uses statistics and programming",
    "machine learning uses data to learn patterns",
    "programming supports data analysis",
    "statistics helps evaluate machine learning models",
]

## 3. Fit TF-IDF and inspect weights

In [3]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"doc_{i}" for i in range(1, len(documents) + 1)],
)

display(tfidf_df.round(3))

,analysis,and,data,evaluate,helps,learn,learning,machine,models,patterns,programming,science,statistics,supports,to,uses
doc_1,0.000,0.484,0.309,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.381,0.484,0.381,0.000,0.000,0.381
doc_2,0.000,0.000,0.278,0.000,0.000,0.436,0.343,0.343,0.000,0.436,0.000,0.000,0.000,0.000,0.436,0.343
doc_3,0.575,0.000,0.367,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.453,0.000,0.000,0.575,0.000,0.000
doc_4,0.000,0.000,0.000,0.453,0.453,0.000,0.357,0.357,0.453,0.000,0.000,0.000,0.357,0.000,0.000,0.000


## 4. Understand term frequency and inverse document frequency

A simplified conceptual form is `TF-IDF(term, document) = TF(term, document) × IDF(term)`. Scikit-learn applies specific conventions for term frequency, smoothing, and normalization; the default output is L2-normalized per document.

In [4]:
terms = vectorizer.get_feature_names_out()
idf_df = pd.DataFrame({
    "term": terms,
    "idf": vectorizer.idf_,
}).sort_values("idf", ascending=False)

display(idf_df.head(15).round(3))

,term,idf
0,analysis,1.916
1,and,1.916
3,evaluate,1.916
4,helps,1.916
5,learn,1.916
11,science,1.916
9,patterns,1.916
8,models,1.916
14,to,1.916
13,supports,1.916


## 5. Compare count and TF-IDF values

Count features represent raw term counts. TF-IDF values reflect relative importance under the fitted corpus and normalization settings.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(vocabulary=vectorizer.vocabulary_)
count_matrix = count_vectorizer.fit_transform(documents)

term = "data"
term_index = list(vectorizer.get_feature_names_out()).index(term)

comparison = pd.DataFrame({
    "document": [f"doc_{i}" for i in range(1, len(documents) + 1)],
    "count": count_matrix.toarray()[:, term_index],
    "tfidf": X.toarray()[:, term_index],
})

display(comparison.round(3))

,document,count,tfidf
0,doc_1,1,0.309
1,doc_2,1,0.278
2,doc_3,1,0.367
3,doc_4,0,0.000


## 6. Use TF-IDF with bigrams

N-grams can represent common phrases as separate features.

In [6]:
ngram_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_ngram = ngram_vectorizer.fit_transform(documents)

print("Feature count with unigrams:", len(TfidfVectorizer().fit(documents).get_feature_names_out()))
print("Feature count with unigrams and bigrams:", len(ngram_vectorizer.get_feature_names_out()))
print("Sample phrase features:", [
    feature for feature in ngram_vectorizer.get_feature_names_out()
    if "machine learning" in feature or "data science" in feature
])

Feature count with unigrams: 16
Feature count with unigrams and bigrams: 34
Sample phrase features: ['data science', 'machine learning']


## 7. Transform unseen documents safely

Fit the vectorizer only on training documents, then use `transform` on new text.

In [7]:
new_text = ["data science and machine learning"]
new_vector = vectorizer.transform(new_text)

new_df = pd.DataFrame(
    new_vector.toarray(),
    columns=vectorizer.get_feature_names_out(),
)
display(new_df.round(3))

,analysis,and,data,evaluate,helps,learn,learning,machine,models,patterns,programming,science,statistics,supports,to,uses
0,0.0,0.523,0.334,0.0,0.0,0.0,0.413,0.413,0.0,0.0,0.0,0.523,0.0,0.0,0.0,0.0


## Key takeaways

- TF-IDF reduces the influence of terms that appear in many documents.
- The numerical values depend on the corpus and vectorizer settings.
- TF-IDF does not understand meaning or word order by itself.
- Fit the vectorizer on training data only to avoid data leakage.